In [20]:
from Imports import *
from Helper import *
from Preprocessing import *
from Plotting import *
%matplotlib inline

In [21]:
# Load dataset and device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
jet_images_path = '../data/jet-images_Mass60-100_pT250-300_R1.25_Pix25.hdf5'
jet_mass_data = HDF5File(jet_images_path, 'r')

print(jet_mass_data.keys())
print(jet_mass_data['image'].shape)

<KeysViewHDF5 ['image', 'jet_delta_R', 'jet_eta', 'jet_mass', 'jet_phi', 'jet_pt', 'signal', 'tau_1', 'tau_2', 'tau_21', 'tau_3', 'tau_32']>
(872666, 25, 25)


In [22]:
# Define variable and dataset
batch_size = 256
n_events = int(.5 * jet_mass_data['image'].shape[0])

latent_dim = 256
lr = 1e-3
n_epochs = 100
num = 4

dataset = JetDataset(jet_mass_data, n_events)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

print("Number of samples:", len(dataset))
print("Image shape:", dataset.images.shape)
print("Feature shape:", dataset.features.shape)

Image shape: torch.Size([436333, 16, 16])
dR Mean: torch.Size([436333])
dR STD: torch.Size([436333])
Pixel Mean: torch.Size([436333])
Pixel STD: torch.Size([436333])
ΔR min: 0.0
ΔR max: 2.4713454246520996
ΔR mean min: 0.0037917911540716887
ΔR mean max: 0.015763528645038605
ΔR std min: 0.017520317807793617
ΔR std max: 0.16022197902202606
Weights (pixel intensity) min: 0.0
Weights (pixel intensity) max: 1.0
Pixel mean min: 0.0028264394495636225
Pixel mean max: 0.00432002916932106
Pixel std min: 0.008714092895388603
Pixel std max: 0.06249340996146202
Number of samples: 436333
Image shape: torch.Size([436333, 16, 16])
Feature shape: torch.Size([436333, 9])


In [23]:
def smotenc_oversample(data, n_samples, categorical_features=None, batch_size=50000, random_state=42):
    rng = np.random.default_rng(random_state)
    data = np.array(data)
    n_events, n_features = data.shape

    # Initialize result with the real data
    minority_all = [data]
    minority = []

    # Calculate how many synthetic samples
    needed = n_samples
    smote = SMOTENC(
        categorical_features=categorical_features if categorical_features else [],
        random_state=random_state,
        k_neighbors=3  # smaller = faster + less memory
    )

    all_X_data = np.empty([needed, 9])
    
    i = 0
    while needed > 0:
        # step = min(needed, batch_size)
        step = needed
        dummy_placeholders = np.ones((len(data) + step, n_features)) * -999

        print(needed)
        
        labels = np.hstack([np.ones(len(data)), np.zeros(len(dummy_placeholders))])
        combined_data = np.vstack([data, dummy_placeholders])
        X_tmp, y_tmp = smote.fit_resample(combined_data, labels)
    
        X_tmp = X_tmp[y_tmp == 1]
        X_tmp = X_tmp[len(data):]
    
        all_X_data[i:i+step] = X_tmp
        
        i += step
        needed -= step
        
    print(all_X_data.shape)

    return all_X_data

real_data = dataset.features
minority = smotenc_oversample(real_data, n_samples=900000, categorical_features=[0])

900000
(900000, 9)


In [11]:
print(real_data.shape)

torch.Size([87266, 9])


In [ ]:
# Run oversampling
real_data = dataset.features
minority = smotenc_oversample(real_data, n_samples=2000, categorical_features=[0])

# --- Plot each feature distribution ---
n_features = real_data.shape[1]
fig, axes = plt.subplots(n_features, 1, figsize=(8, 3*n_features))

minority = torch.tensor(minority)
print(minority.shape)
for i in range(n_features):
    ax = axes[i] if n_features > 1 else axes
    ax.hist(real_data[:, i], bins=50, alpha=0.6, label="Real (minority)")
    ax.hist(minority[:, i], bins=50, alpha=0.6, label="SMOTENC (oversampled)")
    ax.set_title(f"Feature {i}")
    ax.legend()

plt.tight_layout()
plt.show()

In [38]:
# ----------------------------
# 1. Suppose you already have a dataset (minority class)
# ----------------------------
# Example: 10 samples, 5 features
X_minority = torch.tensor([
    [0, 1.2, 1, 3.4, 5.6],
    [1, 0.8, 0, 2.2, 4.5],
    [0, 1.5, 1, 3.1, 5.0],
    [1, 0.7, 0, 2.4, 4.9],
    [0, 1.1, 1, 3.5, 5.2],
    [1, 0.9, 0, 2.1, 4.7],
    [0, 1.3, 1, 3.2, 5.3],
    [1, 0.6, 0, 2.0, 4.6],
    [0, 1.4, 1, 3.6, 5.8],
    [1, 0.5, 0, 2.3, 4.4],
], dtype=torch.float)

y_minority = torch.ones(X_minority.size(0), dtype=torch.long)  # Label = 1

# ----------------------------
# 2. Create dummy majority class (label = 0)
# ----------------------------
# Here we’ll just sample random values in the same feature space
X_majority = torch.randn_like(X_minority) * 2 + 5.0  # different distribution
y_majority = torch.zeros(X_majority.size(0), dtype=torch.long)

# ----------------------------
# 3. Combine into one dataset
# ----------------------------
X = torch.cat([X_minority, X_majority], dim=0)
y = torch.cat([y_minority, y_majority], dim=0)

print("Before resampling:", X.shape, y.shape)
print("Class counts:", torch.bincount(y))

# ----------------------------
# 4. Convert to numpy
# ----------------------------
X_np, y_np = X.numpy(), y.numpy()

# ----------------------------
# 5. Apply SMOTENC
# ----------------------------
categorical_features = [0, 2]  # indices of categorical columns
smote_nc = SMOTENC(categorical_features=categorical_features, random_state=42)

X_resampled, y_resampled = smote_nc.fit_resample(X_np, y_np)

print("After resampling:", X_resampled.shape, y_resampled.shape)
print("Resampled class distribution:", np.bincount(y_resampled))

# ----------------------------
# 6. Convert back to PyTorch
# ----------------------------
X_resampled = torch.tensor(X_resampled, dtype=torch.float)
y_resampled = torch.tensor(y_resampled, dtype=torch.long)

print("Back to tensors:", X_resampled.shape, y_resampled.shape)


Before resampling: torch.Size([20, 5]) torch.Size([20])
Class counts: tensor([10, 10])
After resampling: (20, 5) (20,)
Resampled class distribution: [10 10]
Back to tensors: torch.Size([20, 5]) torch.Size([20])
